# Few-shot damaged-module synthesis with Stable Diffusion inpainting

Generates synthetic **damaged battery module crops** for the condition classifier by:
1. LoRA fine-tuning Stable Diffusion on the ~16 real *bad* crops (few-shot, DreamBooth-style)
2. Inpainting damage into masked regions of *good* crops with the fine-tuned model
3. Zipping the results for download into `data/classifier/bad_synth/`

**Runtime:** Colab free tier, GPU (Runtime → Change runtime type → T4 GPU).
Fine-tuning ≈ 15–30 min; generation ≈ 2–4 s/image.

**Method background:** stableIDG ([PMC12115093](https://pmc.ncbi.nlm.nih.gov/articles/PMC12115093/)),
SynSur / ControlNet inpainting ([JIM 2026](https://link.springer.com/article/10.1007/s10845-026-02915-2)),
survey: [Awesome-Few-Shot-Defect-Image-Generation](https://github.com/bcmi/Awesome-Few-Shot-Defect-Image-Generation).

**Quality control (important):** review every generated crop, delete implausible ones,
keep synthetic ≤50% of the bad class, and only ever evaluate on the real held-out test set.

In [ ]:
# 1. Install dependencies (Colab)
!pip install -q diffusers[torch] transformers accelerate peft safetensors

import torch
assert torch.cuda.is_available(), "Enable GPU: Runtime -> Change runtime type -> T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

## 2. Upload training data

Create two zips locally from the project and upload when prompted:

```bash
# On your machine, from the project root:
zip -r bad_crops.zip  data/classifier/train/bad
zip -r good_crops.zip data/classifier/train/good
```

In [ ]:
from google.colab import files
import zipfile, pathlib, shutil

for name, dest in [("bad_crops.zip", "bad"), ("good_crops.zip", "good")]:
    print(f"Upload {name}:")
    up = files.upload()
    d = pathlib.Path(dest); shutil.rmtree(d, ignore_errors=True); d.mkdir()
    with zipfile.ZipFile(next(iter(up))) as z:
        for m in z.namelist():
            if m.lower().endswith((".jpg", ".jpeg", ".png")):
                (d / pathlib.Path(m).name).write_bytes(z.read(m))
    print(f"  {len(list(d.iterdir()))} images -> {dest}/")

In [ ]:
# 3. LoRA fine-tune SD 1.5 on the real bad crops (DreamBooth-style, few-shot)
# The rare-token prompt 'sks damaged battery module' binds the damage appearance.
!wget -q https://raw.githubusercontent.com/huggingface/diffusers/main/examples/dreambooth/train_dreambooth_lora.py

!accelerate launch train_dreambooth_lora.py \
  --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
  --instance_data_dir="bad" \
  --instance_prompt="a photo of sks damaged battery module, corrosion, burn marks" \
  --output_dir="lora_damage" \
  --resolution=512 --train_batch_size=1 \
  --learning_rate=1e-4 --lr_scheduler="constant" --lr_warmup_steps=0 \
  --max_train_steps=800 --checkpointing_steps=800 \
  --gradient_accumulation_steps=1 --mixed_precision="fp16" --seed=0

In [ ]:
# 4. Load the inpainting pipeline and attach the damage LoRA
from diffusers import StableDiffusionInpaintPipeline
import torch

pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting", torch_dtype=torch.float16,
    safety_checker=None,
).to("cuda")
pipe.load_lora_weights("lora_damage")
print("Pipeline ready.")

In [ ]:
# 5. Batch-generate: inpaint damage into random regions of good crops
import random, pathlib
import numpy as np
from PIL import Image, ImageDraw

PROMPT   = "a photo of sks damaged battery module, corrosion, burn marks, scratches"
NEGATIVE = "cartoon, drawing, text, watermark, blurry, deformed"
N_PER_GOOD = 3          # variants per good crop
random.seed(0)

out_dir = pathlib.Path("bad_synth"); out_dir.mkdir(exist_ok=True)
good_imgs = sorted(pathlib.Path("good").iterdir())

def random_mask(size=512):
    """1-2 random ellipse regions to inpaint (white = regenerate)."""
    m = Image.new("L", (size, size), 0)
    d = ImageDraw.Draw(m)
    for _ in range(random.randint(1, 2)):
        x, y = random.randint(0, size - 1), random.randint(0, size - 1)
        rx, ry = random.randint(size // 8, size // 3), random.randint(size // 8, size // 3)
        d.ellipse([x - rx, y - ry, x + rx, y + ry], fill=255)
    return m

count = 0
for p in good_imgs:
    img = Image.open(p).convert("RGB").resize((512, 512))
    for i in range(N_PER_GOOD):
        result = pipe(prompt=PROMPT, negative_prompt=NEGATIVE,
                      image=img, mask_image=random_mask(),
                      num_inference_steps=30, strength=0.95,
                      guidance_scale=7.5).images[0]
        result.save(out_dir / f"{p.stem}_sd{i:02d}.jpg", quality=95)
        count += 1
    print(f"{p.name}: {N_PER_GOOD} variants ({count} total)")
print(f"\nDone: {count} synthetic bad crops in {out_dir}/")

In [ ]:
# 6. Preview a sample grid before downloading
import matplotlib.pyplot as plt
from PIL import Image
import pathlib, random

samples = random.sample(sorted(pathlib.Path("bad_synth").iterdir()), k=min(9, count))
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for ax, p in zip(axes.flat, samples):
    ax.imshow(Image.open(p)); ax.set_title(p.name, fontsize=6); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# 7. Zip and download
import shutil
from google.colab import files

shutil.make_archive("bad_synth", "zip", "bad_synth")
files.download("bad_synth.zip")

## 8. Back on your machine

```bash
unzip bad_synth.zip -d data/classifier/bad_synth/
# 1. Review every image; delete implausible ones.
# 2. Copy a curated subset into data/classifier/train/bad/ (synthetic <= 50% of class).
# 3. Retrain and evaluate on the REAL-only test set:
python scripts/train_classifier.py
python scripts/evaluate.py --skip_detector
```